In [1]:
#!pip install ortools
import numpy as np
import pandas as pd
import urllib.request

from ortools.linear_solver import pywraplp
from ortools.sat.python import cp_model

from IPython.display import display

In [2]:
# Escolher instancia. Pode ser de 1 a 20
instancia = 9

# Escreve o arquivo LP
writelp = False

## Leitura dos Dados

In [3]:
url = f"https://raw.githubusercontent.com/mercadolibre/challenge-sbpo-2025/master/datasets/a/instance_{instancia:04}.txt"

# Lista maior para armazenar todas as listas
lista_maior = []

with urllib.request.urlopen(url) as f:
        texto = f.read().decode('utf-8')  # Decodificar para UTF-8
        lista_maior = []
        for line in texto.split('\n'):
          if line:
            # Converter os números da linha para inteiros e armazenar em uma lista
            lista_numeros = list(map(int, line.split()))
            # Adicionar a lista de números na lista maior
            lista_maior.append(lista_numeros)

In [4]:
n_o = lista_maior[0][0] # numero de pedidos
n_i = lista_maior[0][1] # numero de itens
n_a = lista_maior[0][2] # numero de corredores

print(f'numero de pedidos: {n_o}')
print(f'numero de itens: {n_i}')
print(f'numero de corredores: {n_a}')

numero de pedidos: 70
numero de itens: 222
numero de corredores: 304


In [5]:
# Pedidos
pedidos_dict = {}
for u in range(0, n_o):
  pedidos_dict [u] = { p: 0 for p in range(0, n_i) }

for i in range(1, n_o+1):
  nitens = lista_maior[i][0]
  pedido = i - 1
  for j in range(1, nitens+1):
    item = lista_maior[i][2*j-1]
    qtd_item = lista_maior[i][2*j]
    pedidos_dict[pedido][item] = qtd_item
df_ped = pd.DataFrame.from_dict(pedidos_dict,orient='index')

In [6]:
# Corredores
corredores_dict = {}
for u in range(0, n_a):
  corredores_dict[u] = { p: 0 for p in range(0, n_i) }

for i in range(n_o+1, n_o + n_a + 1):
  nitens = lista_maior[i][0]
  corredor = i - n_o - 1
  for j in range(1, nitens+1):
    item = lista_maior[i][2*j-1]
    qtd_item = lista_maior[i][2*j]
    corredores_dict[corredor][item] = qtd_item
df_corr = pd.DataFrame.from_dict(corredores_dict,orient='index')
# Bounds
LB, UB =  lista_maior[ n_o + n_a+1][0], lista_maior[ n_o + n_a+1][1],

## Construção dos Conjuntos

In [7]:
# Lista com todos os itens
I = [p for p in range(0, n_i)]
# Conjunto dos corredores
A = [p for p in range(0, n_a)]
# Conjunto dos pedidos
O = [p for p in range(0, n_o)]

In [8]:
# Contador total de itens em cada pedido
cont_O = {}
for o in pedidos_dict:
  cont_O[o] = sum (pedidos_dict[o][i] for i in I)

In [9]:
# Contador de itens em cada pedido
Io = {}
for u in pedidos_dict:
  Io[u] = [k for k, v in pedidos_dict[u].items() if v > 0]


In [10]:
Uai_lim = {}
for a in A:
  Uai_lim[a] = sum (corredores_dict[a][i] for i in I)

In [11]:
# Lista dos corredores de cada item
Ai = {}
for u in corredores_dict:
  Ai[u] = [k for k, v in corredores_dict[u].items() if v > 0]

## Construção do Modelo

In [12]:
# Create the solver instance
solver = pywraplp.Solver.CreateSolver('CBC')

#### Variaveis de decisao

In [13]:
# Variavel de transporte
X = {}
for o in O:
    for i in I:
        for a in A:
            X[(i, o, a)] = solver.IntVar(0, UB,f"x_{i}_{o}_{a}")

In [14]:
# Variavel que habilita o pedido
Yo = {}
for o in O:
   Yo[o] = solver.BoolVar(f"Yo_{o}")

In [15]:
# Variavel que habilita o corredor
Ya = {}
for a in A:
   Ya[a] = solver.BoolVar(f"Ya_{a}")

#### Restrições

In [16]:
# A soma do que é recolhido em dotos os pedidos precisa estar entre o LB e o UN
solver.Add(sum(cont_O[o]*Yo[o] for o in O) >=  LB,"Pedido_Minimo")
solver.Add(sum(cont_O[o]*Yo[o] for o in O) <= UB,"Pedido_Maximo")

<ortools.linear_solver.pywraplp.Constraint; proxy of <Swig Object of type 'operations_research::MPConstraint *' at 0x78b62f38ccf0> >

In [17]:
# Avalia quando um corredor é utilizado
for a in A:
    parcelas = []
    for o in O:
        for i in Io[o]:
            parcelas.append(X[(i, o, a)])
    #solver.Add((sum(parcelas) <= Uai_lim[a] * Ya[a]),f"Aisle_{a}")

In [18]:
# quando um pedido é ativado, todos os itens dele são recollhidos
for o in O:
    parcelas = []
    for i in I:
        for a in A:
            parcelas.append(X[(i, o, a)])
    solver.Add((sum(parcelas) == cont_O[o] * Yo[o]),f"Order_{o}")

In [19]:
# o recolhimento precisa respeitar a capacidade dos corredores item a item
for a in A:
    for i in I:
        parcelas = []
        for o in O:
            parcelas.append(X[(i, o, a)])
        solver.Add((sum(parcelas) <= corredores_dict[a][i]*Ya[a] ),f"Limite_Item_Aisle_{i}_{a}")

#### Função Objetivo

In [20]:
objective_terms = []
for o in O:
   objective_terms.append(cont_O[o]*Yo[o])
for a in A:
   objective_terms.append(-Uai_lim[a]*Ya[a])
solver.Maximize(sum(objective_terms))

In [21]:
#objective_terms = []
#for o in O:
#   objective_terms.append(cont_O[o]*Yo[o])

#objective_terms2 = []
#for a in A:
#   objective_terms2.append(-Uai_lim[a]*Ya[a])
#solver.Maximize(sum(objective_terms)/sum(objective_terms2))


In [22]:
if writelp:
    lp = solver.ExportModelAsLpFormat(False)
    arquivo = open(f'modelo_{instancia}.lp', 'w')
    arquivo.write(lp)

## Resolve o Modelo

In [23]:
status = solver.Solve()

In [24]:
if status == pywraplp.Solver.OPTIMAL:
    print("Optimal solution found:")
    optimal_cost = solver.Objective().Value()
    print(f"\nTotal Cost: {optimal_cost:.2f} BRL")

    # Calculate total cost for workers and overtime separately
    Yo_sol = {}
    Yo_sol = {u:Yo[u].solution_value() for u in Yo}

    Ya_sol = {}
    Ya_sol = {u:Ya[u].solution_value() for u in A}

    X_sol = {}
    X_sol = {u: X[u].solution_value() for u in X}

    Pedidos_Escolhidos = [ u for u in Yo if Yo[u].solution_value() > 0 ]
    Corredores_Utilizados = [ u  for u in A if Ya[u].solution_value() > 0 ]
    Transportes = { u:X[u].solution_value()  for u in X if X[u].solution_value()>0 }
    df_transp = pd.DataFrame([(u[0], u[1], u[2], Transportes[u]) for u in list(Transportes.keys())],columns=['Item','Pedido','Corredor','Quantidade']).sort_values(by=['Corredor','Pedido'])
    df_transp['corr_item'] =  df_transp['Item'].astype('str')  + '_' + df_transp['Corredor'].astype('str')

    uso_corredores = df_transp[['Item','Pedido','Corredor','Quantidade','corr_item']].pivot_table(index='corr_item',values='Quantidade',aggfunc='sum').reset_index()
    df_corr_melted = df_corr.melt(ignore_index=False).reset_index()
    df_corr_melted.columns = ['corredor', 'item', 'valor']
    df_corr_melted['corr_item'] =  df_corr_melted['item'].astype('str')  + '_' + df_corr_melted['corredor'].astype('str')
    uso_corredores = uso_corredores.merge(df_corr_melted, on='corr_item',how='left')
    uso_corredores['sobra'] = uso_corredores['valor'] - uso_corredores['Quantidade']
    sobras = uso_corredores.sobra.sum() # sobra > = 0 => OK ;
                                        # sobra < 0 => not ok

else:
    print("No optimal solution found.")

Optimal solution found:

Total Cost: -0.00 BRL


In [25]:
def totais_corredores(Transportes):
    tots = {}
    for a in A:
        soma_a = 0
        for u in Transportes:
            (ii,oo,aa) = u
            if aa == a:
                soma_a = soma_a + Transportes[u]
        tots[a] = soma_a
    return tots

In [26]:
def tot_pedidos(Transportes):
    soma = 0
    for u in Transportes:
       soma = soma +Transportes[u]
    return soma

In [27]:
def metrica(Transportes,Corredores_Utilizados):
    return tot_pedidos(Transportes) / len(Corredores_Utilizados)

In [28]:
if status == pywraplp.Solver.OPTIMAL:
    print(f'Instância: {instancia}')
    print(f'numero de itens: {n_i}')
    print(f'numero de pedidos: {n_o}')
    print(f'numero de corredores: {n_a}')
    print(f'Pedido Minimo: {LB}')
    print(f'Pedido Maximo: {UB}')
    print(f'Pedidos Escolhidos: {Pedidos_Escolhidos}')
    print(f'Corredores_Utilizados: {Corredores_Utilizados}')
    print(f'Total de itens da wave: {int(tot_pedidos(Transportes))}')
    print(f'Qauntidade de Pedidos: {len(Pedidos_Escolhidos)}')
    print(f'Total de corredores na wave: {len(Corredores_Utilizados)}')
    print(f'Métrica Challenge: {metrica(Transportes,Corredores_Utilizados)}')
    print(f'Sobras: {sobras}')
    display(df_transp)


Instância: 9
numero de itens: 222
numero de pedidos: 70
numero de corredores: 304
Pedido Minimo: 52
Pedido Maximo: 153
Pedidos Escolhidos: [1, 2, 4, 5, 9, 14, 15, 20, 23, 32, 34, 35, 36, 43, 44, 49, 52, 53, 56, 57, 59, 61, 62, 66, 69]
Corredores_Utilizados: [13, 24, 26, 41, 78, 81, 127, 158, 193, 213, 252, 257, 278, 279, 300]
Total de itens da wave: 135
Qauntidade de Pedidos: 25
Total de corredores na wave: 15
Métrica Challenge: 9.0
Sobras: 0.0


,Item,Pedido,Corredor,Quantidade,corr_item
6,24,5,13,1.0,24_13
13,50,9,13,1.0,50_13
17,73,14,13,1.0,73_13
38,205,43,13,2.0,205_13
5,156,4,24,1.0,156_24
...,...,...,...,...,...
21,158,20,300,1.0,158_300
32,114,35,300,1.0,114_300
36,32,43,300,1.0,32_300
41,114,44,300,2.0,114_300


In [29]:
UB

153

In [30]:
LB

52